# 10 - Homologación monetaria ENIGH 2018-2024

Esta libreta cierra la etapa de homologación monetaria del proyecto. El objetivo es conservar los montos nominales originales y crear versiones comparables en **pesos de 2024** para interpretar cambios reales entre 2018, 2020, 2022 y 2024.

La prioridad es metodológica y escolar: cada tabla debe dejar claro el universo, la unidad de observación, el ponderador y la variable usada. No se estiman modelos, no se recalcula el notebook 09 y no se modifica `data/raw/` ni `data/interim/revision_4/`.

## Metodología oficial revisada

Fuentes oficiales revisadas para esta etapa:

- INEGI, ENIGH 2024, página del programa: objetivo, unidad de observación, ponderador y periodo de levantamiento.
- INEGI, ENIGH 2024, presentación de resultados: publica series 2016-2024 del ingreso corriente promedio trimestral del hogar en **pesos de 2024**.
- INEGI, ENIGH 2024, diseño conceptual: define ingreso corriente, componentes y periodos de referencia de ingreso.
- INEGI, INPC 2024: confirma que el INPC es el indicador oficial de inflación, con base segunda quincena de julio de 2018 = 100 y actualización 2024.
- CONEVAL, líneas de pobreza por ingresos: confirma uso mensual del INPC publicado por INEGI para actualizar referentes monetarios.

La documentación oficial confirma que los resultados históricos publicados por INEGI para ENIGH se presentan en pesos de 2024, pero los marts actuales están agregados a nivel hogar/persona y no conservan el detalle mensual por fuente de ingreso necesario para reproducir una deflactación mensual por componente. Por eso este notebook utiliza una aproximación explícita y validable: un `deflactor_2024` común por año, calibrado contra el benchmark oficial de INEGI para ingreso corriente promedio trimestral del hogar.

Esta decisión evita usar un INPC promedio anual arbitrario. La fórmula operativa es:

```text
deflactor_2024_año = benchmark_inegi_ingreso_corriente_hogar_pesos_2024_año / promedio_nominal_ponderado_mart_hogar_año
monto_real_2024 = monto_nominal * deflactor_2024_año
```

Para 2024 se fija `deflactor_2024 = 1.0`, de modo que los montos nominales y reales coinciden salvo redondeos de publicación externa.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)
    class Markdown(str):
        pass

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

REV4 = ROOT / "data" / "interim" / "revision_4"
REV5 = ROOT / "data" / "interim" / "revision_5"
DOCS = ROOT / "docs"
FIG_DIR = ROOT / "reports" / "figures_documentacion"
REV5.mkdir(parents=True, exist_ok=True)
DOCS.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

HOGAR_FILE = "mart_hogar_2018_2024.csv.gz"
PERSONA_FILE = "mart_persona_2018_2024.csv.gz"
HOGAR_PATH = REV4 / HOGAR_FILE
PERSONA_PATH = REV4 / PERSONA_FILE
ZM_MAP_PATH = DOCS / "zonas_metropolitanas_prioritarias_2020.csv"
DEF_DOC_PATH = DOCS / "deflactores_precios_2024.csv"
DEF_REV5_PATH = REV5 / "deflactores_precios_2024.csv"
SUMMARY_PATH = REV5 / "resumen_homologacion_monetaria.json"

YEARS = [2018, 2020, 2022, 2024]
WEIGHT = "factor"
HH_INC = "ing_cor_hogar_oficial_tri"
HH_PC = "ing_cor_pc_oficial_tri"
PC_PERSONA = "ing_cor_hogar_pc_oficial_tri"
LAB_PERSONA = "ingreso_persona_laboral_negocio_tri"
REGION_ORDER = ["Norte", "Centro Norte", "Centro", "Sur"]
SOCIO_ORDER = ["Bajo", "Medio bajo", "Medio alto", "Alto"]
URBAN_PATTERN = "100 000"
RURAL_PATTERN = "menos de 2 500"
ZONAS_PRIORITARIAS = ["Valle de México", "Guadalajara", "Monterrey"]
SMALL_LOW_GROUP = "Localidad pequeña y estrato socioeconómico bajo"

INEGI_BENCHMARK_2024 = {
    2018: 67319.0,
    2020: 63400.0,
    2022: 70391.0,
    2024: 77864.0,
}

OFFICIAL_SOURCES = {
    "enigh_2024_programa": "https://www.inegi.org.mx/programas/enigh/nc/2024/default.html",
    "enigh_2024_presentacion": "https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/doc/enigh2024_ns_presentacion_resultados.pdf",
    "enigh_2024_diseno_conceptual": "https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/doc/889463924487.pdf",
    "inpc_2024_metadatos": "https://www.inegi.org.mx/rnm/index.php/catalog/1015",
    "coneval_lpi": "https://www.coneval.org.mx/Medicion/MP/Paginas/Lineas_Pobreza_Ingresos_Serie_1992-2024.aspx",
}

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})

print(f"ROOT: {ROOT}")
print(f"Entrada revision_4: {REV4}")
print(f"Salida revision_5: {REV5}")

## Funciones de estimación

Todas las estimaciones descriptivas ponderadas usan `factor`, no `factor * tot_integ`, cuando la base ya está en filas de persona. Para hogares, la unidad es hogar y el ponderador también es `factor`.

In [ ]:
def numeric(series):
    return pd.to_numeric(series, errors="coerce")


def valid_weighted_mask(values, weights, nonnegative=True, positive_only=False):
    values = numeric(values)
    weights = numeric(weights)
    mask = values.notna() & weights.notna() & weights.gt(0)
    if nonnegative:
        mask &= values.ge(0)
    if positive_only:
        mask &= values.gt(0)
    return values, weights, mask


def weighted_mean(values, weights, positive_only=False):
    values, weights, mask = valid_weighted_mask(values, weights, positive_only=positive_only)
    if not mask.any():
        return np.nan
    return float(np.average(values[mask], weights=weights[mask]))


def weighted_quantile(values, weights, quantiles, positive_only=False):
    values, weights, mask = valid_weighted_mask(values, weights, positive_only=positive_only)
    values = values[mask].to_numpy(dtype=float)
    weights = weights[mask].to_numpy(dtype=float)
    if values.size == 0 or weights.sum() <= 0:
        return [np.nan for _ in quantiles]
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    cumulative = np.cumsum(weights)
    cutoff = np.array(quantiles, dtype=float) * weights.sum()
    return [float(values[np.searchsorted(cumulative, q, side="left")]) for q in cutoff]


def weighted_gini(values, weights, positive_only=False):
    values, weights, mask = valid_weighted_mask(values, weights, positive_only=positive_only)
    x = values[mask].to_numpy(dtype=float)
    w = weights[mask].to_numpy(dtype=float)
    if x.size == 0 or w.sum() <= 0 or np.sum(x * w) <= 0:
        return np.nan
    order = np.argsort(x)
    x = x[order]
    w = w[order]
    cumw = np.cumsum(w)
    cumxw = np.cumsum(x * w)
    population_share = np.concatenate(([0.0], cumw / cumw[-1]))
    income_share = np.concatenate(([0.0], cumxw / cumxw[-1]))
    return float(1 - 2 * np.trapezoid(income_share, population_share))


def weighted_summary(df, group_cols, value_col, weight_col=WEIGHT, positive_only=False, min_n=0):
    group_cols = [group_cols] if isinstance(group_cols, str) else list(group_cols)
    needed = group_cols + [value_col, weight_col]
    data = df[needed].copy()
    data[value_col] = numeric(data[value_col])
    data[weight_col] = numeric(data[weight_col])
    data = data[data[value_col].notna() & data[weight_col].gt(0)]
    data = data[data[value_col].ge(0)]
    if positive_only:
        data = data[data[value_col].gt(0)]
    for col in group_cols:
        data[col] = data[col].astype("object").where(data[col].notna(), "Sin dato / no aplica")
    rows = []
    for key, group in data.groupby(group_cols, dropna=False, sort=False):
        key_tuple = key if isinstance(key, tuple) else (key,)
        q10, q25, q50, q75, q90, q95 = weighted_quantile(group[value_col], group[weight_col], [0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        row = dict(zip(group_cols, key_tuple))
        row.update({
            "n_muestral": int(len(group)),
            "poblacion_expandida": float(group[weight_col].sum()),
            "media_ponderada": weighted_mean(group[value_col], group[weight_col]),
            "p10": q10,
            "p25": q25,
            "mediana_ponderada": q50,
            "p75": q75,
            "p90": q90,
            "p95": q95,
            "gini_ponderado_0_100": 100 * weighted_gini(group[value_col], group[weight_col]),
        })
        rows.append(row)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out = out[out["n_muestral"].ge(min_n)].copy()
    return out.reset_index(drop=True)


def as_code(series, width):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.zfill(width)
    )


def pct_change(start, end):
    if pd.isna(start) or pd.isna(end) or start == 0:
        return np.nan
    return float((end / start - 1) * 100)


def money(x):
    if pd.isna(x):
        return ""
    return f"${x:,.0f}"


def pct(x):
    if pd.isna(x):
        return ""
    return f"{x:,.1f}%"


def pp(x):
    if pd.isna(x):
        return ""
    return f"{x:,.1f} pp"


def pretty_money_table(df, money_cols=None, pct_cols=None, pp_cols=None, int_cols=None):
    out = df.copy()
    money_cols = money_cols or []
    pct_cols = pct_cols or []
    pp_cols = pp_cols or []
    int_cols = int_cols or []
    for col in money_cols:
        if col in out.columns:
            out[col] = out[col].map(money)
    for col in pct_cols:
        if col in out.columns:
            out[col] = out[col].map(pct)
    for col in pp_cols:
        if col in out.columns:
            out[col] = out[col].map(pp)
    for col in int_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: "" if pd.isna(x) else f"{int(round(x)):,.0f}")
    return out


def save_doc_figure(fig, filename, alt_text):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"![{alt_text}](../reports/figures_documentacion/{filename})"))
    return str(path)


def clean_for_json(obj):
    if obj is pd.NA:
        return None
    if isinstance(obj, dict):
        return {str(k): clean_for_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, tuple):
        return [clean_for_json(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        if np.isnan(obj):
            return None
        return float(obj)
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj

## Carga e inventario monetario

Se leen los marts de `revision_4` completos porque `revision_5` debe preservar las columnas nominales y solo añadir variables reales y metadata de homologación.

In [ ]:
hogar = pd.read_csv(HOGAR_PATH, low_memory=False)
persona = pd.read_csv(PERSONA_PATH, low_memory=False)

expected_rows = {
    "mart_hogar": 345_169,
    "mart_persona": 1_203_231,
}

assert len(hogar) == expected_rows["mart_hogar"], f"Filas hogar inesperadas: {len(hogar):,}"
assert len(persona) == expected_rows["mart_persona"], f"Filas persona inesperadas: {len(persona):,}"

hogar_money_cols = [
    "ing_cor_hogar_oficial_tri",
    "ingtrab_hogar_oficial_tri",
    "trabajo_subordinado_hogar_tri",
    "sueldos_hogar_tri",
    "negocio_hogar_tri",
    "rentas_hogar_tri",
    "transfer_hogar_tri",
    "otros_ing_hogar_tri",
    "gasto_mon_hogar_tri",
    "ingreso_personas_total_registros_tri",
    "ingreso_personas_laboral_negocio_tri",
    "ingreso_personas_transferencias_tri",
    "ingreso_personas_rentas_propiedad_tri",
    "ingreso_personas_financiero_capital_tri",
    "ing_cor_pc_oficial_tri",
    "ingtrab_pc_oficial_tri",
]

persona_money_cols = [
    "ingreso_persona_total_registros_tri",
    "ingreso_persona_laboral_negocio_tri",
    "ingreso_persona_rentas_propiedad_tri",
    "ingreso_persona_transferencias_tri",
    "ingreso_persona_financiero_capital_tri",
    "ingreso_persona_no_clasificado_tri",
    "ing_cor_hogar_oficial_tri",
    "ingtrab_hogar_oficial_tri",
    "trabajo_subordinado_hogar_tri",
    "sueldos_hogar_tri",
    "negocio_hogar_tri",
    "rentas_hogar_tri",
    "transfer_hogar_tri",
    "otros_ing_hogar_tri",
    "gasto_mon_hogar_tri",
    "ing_cor_hogar_pc_oficial_tri",
    "ingtrab_hogar_pc_oficial_tri",
]

hogar_money_cols = [c for c in hogar_money_cols if c in hogar.columns]
persona_money_cols = [c for c in persona_money_cols if c in persona.columns]

inventory = pd.concat([
    pd.DataFrame({"mart": "mart_hogar", "variable_nominal": hogar_money_cols}),
    pd.DataFrame({"mart": "mart_persona", "variable_nominal": persona_money_cols}),
], ignore_index=True)
inventory["variable_real_2024"] = inventory["variable_nominal"] + "_real_2024"

display(Markdown("### Inventario de variables monetarias a homologar"))
display(inventory)

print(f"Variables monetarias hogar: {len(hogar_money_cols)}")
print(f"Variables monetarias persona: {len(persona_money_cols)}")

## Tabla explícita de deflactores

El benchmark oficial usado para calibrar el deflactor es el ingreso corriente promedio trimestral del hogar publicado por INEGI en pesos de 2024:

| Año | INEGI |
| --- | ---: |
| 2018 | 67,319 |
| 2020 | 63,400 |
| 2022 | 70,391 |
| 2024 | 77,864 |

La variable comparable del mart es `ing_cor_hogar_oficial_tri`, en la base de hogares y ponderada con `factor`.

In [ ]:
for col in ["anio", WEIGHT, HH_INC] + hogar_money_cols:
    if col in hogar.columns:
        hogar[col] = numeric(hogar[col])
for col in ["anio", WEIGHT] + persona_money_cols:
    if col in persona.columns:
        persona[col] = numeric(persona[col])

mean_nominal_rows = []
for year in YEARS:
    group = hogar[hogar["anio"].eq(year)]
    mean_nominal = weighted_mean(group[HH_INC], group[WEIGHT])
    deflator = 1.0 if year == 2024 else INEGI_BENCHMARK_2024[year] / mean_nominal
    mean_nominal_rows.append({
        "anio": year,
        "indice_o_referencia": "Benchmark INEGI: ingreso corriente promedio trimestral del hogar en pesos de 2024",
        "promedio_nominal_ponderado_mart": mean_nominal,
        "benchmark_inegi_pesos_2024": INEGI_BENCHMARK_2024[year],
        "deflactor_2024": deflator,
        "fuente": "INEGI ENIGH 2024, Presentación de resultados; INPC oficial como marco de precios",
        "metodo": "Deflactor anual por edición calibrado al benchmark oficial; 2024 fijado en 1.0",
    })

deflactor_table = pd.DataFrame(mean_nominal_rows)
deflactor_table.to_csv(DEF_DOC_PATH, index=False, encoding="utf-8")
deflactor_table.to_csv(DEF_REV5_PATH, index=False, encoding="utf-8")

deflactors = dict(zip(deflactor_table["anio"], deflactor_table["deflactor_2024"]))

display(pretty_money_table(
    deflactor_table,
    money_cols=["promedio_nominal_ponderado_mart", "benchmark_inegi_pesos_2024"],
))

## Construcción de variables reales y guardado de `revision_5`

La regla es conservar cada variable nominal y añadir una variable con sufijo `_real_2024`. También se añade `deflactor_2024` como metadata de conversión monetaria. Esta columna no sustituye a `factor`, que sigue siendo el ponderador muestral.

In [ ]:
hogar_rev5 = hogar.copy()
persona_rev5 = persona.copy()

for df, cols in [(hogar_rev5, hogar_money_cols), (persona_rev5, persona_money_cols)]:
    df["deflactor_2024"] = df["anio"].map(deflactors)
    assert df["deflactor_2024"].notna().all(), "Hay filas sin deflactor_2024."
    for col in cols:
        real_col = f"{col}_real_2024"
        df[real_col] = numeric(df[col]) * df["deflactor_2024"]

hogar_rev5.to_csv(REV5 / HOGAR_FILE, index=False, compression="gzip")
persona_rev5.to_csv(REV5 / PERSONA_FILE, index=False, compression="gzip")

real_cols_hogar = [f"{c}_real_2024" for c in hogar_money_cols]
real_cols_persona = [f"{c}_real_2024" for c in persona_money_cols]

print(f"Guardado: {REV5 / HOGAR_FILE}")
print(f"Guardado: {REV5 / PERSONA_FILE}")
print(f"Variables reales hogar: {len(real_cols_hogar)}")
print(f"Variables reales persona: {len(real_cols_persona)}")

## Validación de granularidad y preservación

`revision_5` debe mantener filas, llaves, ponderadores y geografía. Solo se agregan variables reales y `deflactor_2024`.

In [ ]:
def same_series(a, b):
    if pd.api.types.is_numeric_dtype(a) or pd.api.types.is_numeric_dtype(b):
        aa = numeric(a)
        bb = numeric(b)
        return bool(np.allclose(aa.fillna(-999999999), bb.fillna(-999999999)))
    return bool(a.astype("string").fillna("<NA>").equals(b.astype("string").fillna("<NA>")))


validation_rows = []
checks = [
    ("mart_hogar", hogar, hogar_rev5, ["anio", "folioviv", "foliohog"], ["factor", "factor_hogar", "cve_ent", "cve_mun", "region_banxico", "tam_loc_desc", "est_socio_desc"]),
    ("mart_persona", persona, persona_rev5, ["anio", "folioviv", "foliohog", "numren"], ["factor", "factor_hogar", "cve_ent", "cve_mun", "region_banxico", "tam_loc_desc", "est_socio_desc"]),
]

for name, old, new, keys, preserve_cols in checks:
    validation_rows.append({"mart": name, "validacion": "filas", "valor": len(new), "esperado": expected_rows[name], "ok": len(new) == expected_rows[name]})
    validation_rows.append({"mart": name, "validacion": "duplicados_llave", "valor": int(new.duplicated(keys).sum()), "esperado": 0, "ok": int(new.duplicated(keys).sum()) == 0})
    validation_rows.append({"mart": name, "validacion": "llaves_preservadas", "valor": "igual", "esperado": "igual", "ok": old[keys].astype("string").equals(new[keys].astype("string"))})
    for col in preserve_cols:
        if col in old.columns and col in new.columns:
            validation_rows.append({"mart": name, "validacion": f"{col}_preservado", "valor": "igual", "esperado": "igual", "ok": same_series(old[col], new[col])})

validation = pd.DataFrame(validation_rows)
display(validation)
assert validation["ok"].all(), "Alguna validación de granularidad/preservación falló."

max_2024_diffs = []
for name, df, cols in [("mart_hogar", hogar_rev5, hogar_money_cols), ("mart_persona", persona_rev5, persona_money_cols)]:
    mask_2024 = df["anio"].eq(2024)
    for col in cols:
        diff = (numeric(df.loc[mask_2024, f"{col}_real_2024"]) - numeric(df.loc[mask_2024, col])).abs().max()
        max_2024_diffs.append({"mart": name, "variable": col, "max_abs_diff_2024": float(diff) if pd.notna(diff) else np.nan})

same_2024 = pd.DataFrame(max_2024_diffs)
display(same_2024.sort_values("max_abs_diff_2024", ascending=False).head(10))
assert same_2024["max_abs_diff_2024"].fillna(0).max() <= 1e-9, "En 2024 nominal y real no coinciden."

## Validación externa contra INEGI

Se compara nuestro promedio ponderado real de `ing_cor_hogar_oficial_tri_real_2024` contra el benchmark oficial de INEGI. La unidad es hogar, el ponderador es `factor` y la temporalidad es ingreso trimestral.

In [ ]:
HH_INC_REAL = f"{HH_INC}_real_2024"
HH_PC_REAL = f"{HH_PC}_real_2024"
PC_PERSONA_REAL = f"{PC_PERSONA}_real_2024"
LAB_PERSONA_REAL = f"{LAB_PERSONA}_real_2024"

benchmark_rows = []
for year in YEARS:
    group = hogar_rev5[hogar_rev5["anio"].eq(year)]
    nuestro = weighted_mean(group[HH_INC_REAL], group[WEIGHT])
    inegi = INEGI_BENCHMARK_2024[year]
    diff = nuestro - inegi
    benchmark_rows.append({
        "anio": year,
        "nuestro_promedio_real": nuestro,
        "inegi": inegi,
        "diferencia_absoluta": abs(diff),
        "diferencia_pct": 100 * diff / inegi,
        "clasificacion": "reproducción cercana" if abs(diff) <= 1 else "revisar discrepancia",
    })
benchmark_validation = pd.DataFrame(benchmark_rows)
display(pretty_money_table(
    benchmark_validation,
    money_cols=["nuestro_promedio_real", "inegi", "diferencia_absoluta"],
    pct_cols=["diferencia_pct"],
))
assert benchmark_validation["diferencia_absoluta"].max() <= 1.0, "La validación contra INEGI no reproduce el benchmark razonablemente."

## Gini: validación de invariancia

Como el deflactor es común dentro de cada año, el Gini nominal debe coincidir con el Gini real dentro del mismo año. Esta validación no sustituye el notebook 09; solo confirma que la homologación monetaria no cambia medidas relativas intra-año cuando el multiplicador es uniforme.

In [ ]:
gini_rows = []
for year in YEARS:
    group = hogar_rev5[hogar_rev5["anio"].eq(year)]
    g_nom = 100 * weighted_gini(group[HH_INC], group[WEIGHT])
    g_real = 100 * weighted_gini(group[HH_INC_REAL], group[WEIGHT])
    gini_rows.append({
        "anio": year,
        "gini_nominal_0_100": g_nom,
        "gini_real_0_100": g_real,
        "diferencia": g_real - g_nom,
    })
gini_invariance = pd.DataFrame(gini_rows)
display(gini_invariance.round(8))
assert gini_invariance["diferencia"].abs().max() <= 1e-8, "El Gini cambió con un deflactor común anual."

## Comparación nacional nominal vs real

Esta sección muestra por qué era necesaria la homologación: en términos nominales todos los montos tienden a aumentar más, pero la lectura real corrige el cambio de precios.

In [ ]:
national_nom = weighted_summary(hogar_rev5, "anio", HH_INC)
national_real = weighted_summary(hogar_rev5, "anio", HH_INC_REAL)
national = national_nom[["anio", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90"]].merge(
    national_real[["anio", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90"]],
    on="anio",
    suffixes=("_nominal", "_real_2024"),
)

display(pretty_money_table(
    national,
    money_cols=[c for c in national.columns if c != "anio"],
))

periods = [(2018, 2020), (2020, 2022), (2022, 2024), (2018, 2024)]
change_rows = []
for start, end in periods:
    s = national[national["anio"].eq(start)].iloc[0]
    e = national[national["anio"].eq(end)].iloc[0]
    for metric in ["media_ponderada", "mediana_ponderada", "p25", "p75", "p90"]:
        change_rows.append({
            "indicador": metric,
            "periodo": f"{start}-{end}",
            "cambio_nominal_pct": pct_change(s[f"{metric}_nominal"], e[f"{metric}_nominal"]),
            "cambio_real_pct": pct_change(s[f"{metric}_real_2024"], e[f"{metric}_real_2024"]),
        })
national_changes = pd.DataFrame(change_rows)
national_changes["diferencia_pp"] = national_changes["cambio_real_pct"] - national_changes["cambio_nominal_pct"]
display(pretty_money_table(national_changes, pct_cols=["cambio_nominal_pct", "cambio_real_pct"], pp_cols=["diferencia_pp"]))

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 5.2))
ax.plot(national["anio"], national["media_ponderada_nominal"], marker="o", color="#5B8CBA", label="Media nominal")
ax.plot(national["anio"], national["media_ponderada_real_2024"], marker="o", color="#D1495B", label="Media real 2024")
ax.plot(national["anio"], national["mediana_ponderada_nominal"], marker="s", linestyle="--", color="#76B041", label="Mediana nominal")
ax.plot(national["anio"], national["mediana_ponderada_real_2024"], marker="s", linestyle="--", color="#F0A202", label="Mediana real 2024")
ax.set_title("Evolución nominal vs real del ingreso corriente del hogar")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos trimestrales")
ax.set_xticks(YEARS)
ax.legend(frameon=False, ncols=2)
ax.text(0.01, -0.18, "Unidad: hogar. Ponderador: factor. Ingreso corriente trimestral; real expresado en pesos de 2024.", transform=ax.transAxes, fontsize=9)
figures = []
figures.append(save_doc_figure(fig, "ingreso_nacional_nominal_real_2018_2024.png", "Evolución nominal vs real del ingreso corriente del hogar"))

## Impacto de la homologación monetaria

Se comparan cambios nominales y reales de indicadores centrales entre 2018 y 2024. La diferencia entre ambas columnas no se interpreta como efecto causal de inflación; solo muestra cuánto cambia la lectura al expresar montos en pesos constantes.

In [ ]:
def get_year_value(summary_df, year, value_col="mediana_ponderada"):
    row = summary_df[summary_df["anio"].eq(year)]
    if row.empty:
        return np.nan
    return float(row.iloc[0][value_col])


def group_median(summary_df, year, group_col, group_value):
    row = summary_df[summary_df["anio"].eq(year) & summary_df[group_col].astype(str).eq(group_value)]
    if row.empty:
        return np.nan
    return float(row.iloc[0]["mediana_ponderada"])


pc_persona_nom = weighted_summary(persona_rev5, "anio", PC_PERSONA)
pc_persona_real = weighted_summary(persona_rev5, "anio", PC_PERSONA_REAL)
lab_nom = weighted_summary(persona_rev5, "anio", LAB_PERSONA, positive_only=True)
lab_real = weighted_summary(persona_rev5, "anio", LAB_PERSONA_REAL, positive_only=True)

region_pc_nom = weighted_summary(persona_rev5, ["anio", "region_banxico"], PC_PERSONA)
region_pc_real = weighted_summary(persona_rev5, ["anio", "region_banxico"], PC_PERSONA_REAL)
tam_pc_nom = weighted_summary(persona_rev5, ["anio", "tam_loc_desc"], PC_PERSONA)
tam_pc_real = weighted_summary(persona_rev5, ["anio", "tam_loc_desc"], PC_PERSONA_REAL)
socio_pc_nom = weighted_summary(persona_rev5, ["anio", "est_socio_desc"], PC_PERSONA)
socio_pc_real = weighted_summary(persona_rev5, ["anio", "est_socio_desc"], PC_PERSONA_REAL)

urban_label = next(v for v in sorted(persona_rev5["tam_loc_desc"].dropna().astype(str).unique()) if URBAN_PATTERN in v)
rural_label = next(v for v in sorted(persona_rev5["tam_loc_desc"].dropna().astype(str).unique()) if RURAL_PATTERN in v)

indicator_specs = [
    ("Media ingreso corriente hogar", get_year_value(national_nom, 2018, "media_ponderada"), get_year_value(national_nom, 2024, "media_ponderada"), get_year_value(national_real, 2018, "media_ponderada"), get_year_value(national_real, 2024, "media_ponderada")),
    ("Mediana ingreso corriente hogar", get_year_value(national_nom, 2018), get_year_value(national_nom, 2024), get_year_value(national_real, 2018), get_year_value(national_real, 2024)),
    ("Mediana ingreso corriente per cápita", get_year_value(pc_persona_nom, 2018), get_year_value(pc_persona_nom, 2024), get_year_value(pc_persona_real, 2018), get_year_value(pc_persona_real, 2024)),
    ("Mediana ingreso laboral individual positivo", get_year_value(lab_nom, 2018), get_year_value(lab_nom, 2024), get_year_value(lab_real, 2018), get_year_value(lab_real, 2024)),
]

for label, df_nom, df_real, group_col, high, low in [
    ("Brecha Norte-Sur", region_pc_nom, region_pc_real, "region_banxico", "Norte", "Sur"),
    ("Brecha urbano-rural", tam_pc_nom, tam_pc_real, "tam_loc_desc", urban_label, rural_label),
    ("Brecha estrato Alto-Bajo", socio_pc_nom, socio_pc_real, "est_socio_desc", "Alto", "Bajo"),
]:
    n18 = group_median(df_nom, 2018, group_col, high) - group_median(df_nom, 2018, group_col, low)
    n24 = group_median(df_nom, 2024, group_col, high) - group_median(df_nom, 2024, group_col, low)
    r18 = group_median(df_real, 2018, group_col, high) - group_median(df_real, 2018, group_col, low)
    r24 = group_median(df_real, 2024, group_col, high) - group_median(df_real, 2024, group_col, low)
    indicator_specs.append((label, n18, n24, r18, r24))

impact_rows = []
for label, n18, n24, r18, r24 in indicator_specs:
    c_nom = pct_change(n18, n24)
    c_real = pct_change(r18, r24)
    impact_rows.append({
        "indicador": label,
        "periodo": "2018-2024",
        "valor_2018_nominal": n18,
        "valor_2024_nominal": n24,
        "valor_2018_real_2024": r18,
        "valor_2024_real_2024": r24,
        "cambio_nominal_pct": c_nom,
        "cambio_real_pct": c_real,
        "diferencia_atribuible_al_ajuste_pp": c_real - c_nom,
    })
impact_table = pd.DataFrame(impact_rows)
display(pretty_money_table(
    impact_table,
    money_cols=["valor_2018_nominal", "valor_2024_nominal", "valor_2018_real_2024", "valor_2024_real_2024"],
    pct_cols=["cambio_nominal_pct", "cambio_real_pct"],
    pp_cols=["diferencia_atribuible_al_ajuste_pp"],
))

## Evolución real nacional

A partir de aquí, las comparaciones temporales principales usan pesos de 2024.

In [ ]:
national_real_core = national_real[["anio", "media_ponderada", "p25", "mediana_ponderada", "p75", "p90"]].copy()
display(pretty_money_table(national_real_core, money_cols=[c for c in national_real_core.columns if c != "anio"]))

display(Markdown("### Variaciones reales nacionales"))
display(pretty_money_table(
    national_changes[national_changes["indicador"].isin(["media_ponderada", "mediana_ponderada"])][["indicador", "periodo", "cambio_real_pct"]],
    pct_cols=["cambio_real_pct"],
))

## Regiones Banxico en términos reales

Se usa ingreso corriente per cápita del hogar distribuido entre personas: `mart_persona` + `factor`. Esto mantiene consistencia con la decisión metodológica cerrada antes del notebook 10.

In [ ]:
regional_real = weighted_summary(persona_rev5, ["anio", "region_banxico"], PC_PERSONA_REAL)
regional_real["ranking_mediana"] = regional_real.groupby("anio")["mediana_ponderada"].rank(ascending=False, method="dense").astype(int)
regional_real["region_banxico"] = pd.Categorical(regional_real["region_banxico"], categories=REGION_ORDER, ordered=True)
regional_real = regional_real.sort_values(["anio", "region_banxico"])

display(pretty_money_table(
    regional_real[["anio", "region_banxico", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90", "ranking_mediana"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))

region_changes = []
for region in REGION_ORDER:
    vals = regional_real[regional_real["region_banxico"].astype(str).eq(region)].set_index("anio")
    if 2018 in vals.index and 2024 in vals.index:
        region_changes.append({
            "region_banxico": region,
            "mediana_2018_real": vals.loc[2018, "mediana_ponderada"],
            "mediana_2024_real": vals.loc[2024, "mediana_ponderada"],
            "variacion_real_2018_2024_pct": pct_change(vals.loc[2018, "mediana_ponderada"], vals.loc[2024, "mediana_ponderada"]),
        })
region_changes = pd.DataFrame(region_changes)
display(pretty_money_table(region_changes, money_cols=["mediana_2018_real", "mediana_2024_real"], pct_cols=["variacion_real_2018_2024_pct"]))

In [ ]:
plot_region = regional_real.copy()
fig, ax = plt.subplots(figsize=(9.5, 5.2))
colors = {"Norte": "#1D70A2", "Centro Norte": "#7CB518", "Centro": "#F2A541", "Sur": "#C84630"}
for region in REGION_ORDER:
    g = plot_region[plot_region["region_banxico"].astype(str).eq(region)]
    ax.plot(g["anio"], g["mediana_ponderada"], marker="o", label=region, color=colors.get(region))
ax.set_title("Mediana real del ingreso corriente per cápita por región Banxico")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos de 2024, ingreso trimestral per cápita")
ax.set_xticks(YEARS)
ax.legend(frameon=False)
ax.text(0.01, -0.18, "Unidad: persona. Variable: ingreso per cápita del hogar heredado; ponderador: factor.", transform=ax.transAxes, fontsize=9)
figures.append(save_doc_figure(fig, "ingreso_real_region_banxico_2018_2024.png", "Ingreso real por región Banxico"))

## Brecha Norte-Sur real

La brecha se reporta como diferencia absoluta y razón de medianas. La razón Norte/Sur dentro de un mismo año es invariante al deflactor común anual; la diferencia absoluta sí cambia al expresar montos en pesos de 2024.

In [ ]:
def two_group_gap(summary_df, group_col, high_label, low_label):
    rows = []
    for year in YEARS:
        hi = group_median(summary_df, year, group_col, high_label)
        lo = group_median(summary_df, year, group_col, low_label)
        rows.append({
            "anio": year,
            "grupo_alto": high_label,
            "mediana_alta": hi,
            "grupo_bajo": low_label,
            "mediana_baja": lo,
            "diferencia_absoluta": hi - lo,
            "razon": hi / lo if pd.notna(lo) and lo != 0 else np.nan,
        })
    return pd.DataFrame(rows)


north_south_gap = two_group_gap(regional_real, "region_banxico", "Norte", "Sur")
display(pretty_money_table(north_south_gap, money_cols=["mediana_alta", "mediana_baja", "diferencia_absoluta"]))

fig, ax1 = plt.subplots(figsize=(9.5, 5.2))
ax1.bar(north_south_gap["anio"].astype(str), north_south_gap["diferencia_absoluta"], color="#5B8CBA", label="Diferencia absoluta")
ax1.set_ylabel("Diferencia en pesos de 2024")
ax1.set_xlabel("Año")
ax2 = ax1.twinx()
ax2.plot(north_south_gap["anio"].astype(str), north_south_gap["razon"], marker="o", color="#D1495B", label="Razón Norte/Sur")
ax2.set_ylabel("Razón de medianas")
ax1.set_title("Brecha Norte-Sur en ingreso per cápita real")
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, labels + labels2, frameon=False, loc="upper left")
ax1.text(0.01, -0.18, "Unidad: persona. Variable: ingreso corriente per cápita real 2024; ponderador: factor.", transform=ax1.transAxes, fontsize=9)
figures.append(save_doc_figure(fig, "brecha_norte_sur_real_2018_2024.png", "Brecha Norte-Sur real"))

## Tamaño de localidad: 100,000+ vs menos de 2,500 habitantes

Se reporta la evolución real del ingreso per cápita del hogar distribuido entre personas.

In [ ]:
tam_pc_real = tam_pc_real.sort_values(["anio", "tam_loc_desc"])
urban_rural_gap = two_group_gap(tam_pc_real, "tam_loc_desc", urban_label, rural_label)

display(pretty_money_table(
    tam_pc_real[["anio", "tam_loc_desc", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))
display(Markdown("### Brecha urbano-rural real"))
display(pretty_money_table(urban_rural_gap, money_cols=["mediana_alta", "mediana_baja", "diferencia_absoluta"]))

plot_tam = tam_pc_real[tam_pc_real["tam_loc_desc"].isin([urban_label, rural_label])].copy()
fig, ax = plt.subplots(figsize=(9.5, 5.2))
for label, color in [(urban_label, "#1D70A2"), (rural_label, "#F2A541")]:
    g = plot_tam[plot_tam["tam_loc_desc"].eq(label)]
    short = "100,000+ habitantes" if URBAN_PATTERN in label else "Menos de 2,500 habitantes"
    ax.plot(g["anio"], g["mediana_ponderada"], marker="o", label=short, color=color)
ax.set_title("Evolución real del ingreso per cápita por tamaño de localidad")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos de 2024, ingreso trimestral per cápita")
ax.set_xticks(YEARS)
ax.legend(frameon=False)
ax.text(0.01, -0.18, "Unidad: persona. Ponderador: factor. Categorías oficiales de tamaño de localidad ENIGH.", transform=ax.transAxes, fontsize=9)
figures.append(save_doc_figure(fig, "urbano_rural_real_2018_2024.png", "Evolución urbano-rural real"))

## Estrato socioeconómico: Alto vs Bajo

La variable `est_socio_desc` es oficial de INEGI y ya quedó normalizada en etapas previas. Aquí no se interpreta como causal ni como índice de marginación.

In [ ]:
socio_pc_real["est_socio_desc"] = pd.Categorical(socio_pc_real["est_socio_desc"], categories=SOCIO_ORDER, ordered=True)
socio_pc_real = socio_pc_real.sort_values(["anio", "est_socio_desc"])
socio_gap = two_group_gap(socio_pc_real, "est_socio_desc", "Alto", "Bajo")

display(pretty_money_table(
    socio_pc_real[["anio", "est_socio_desc", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))
display(Markdown("### Brecha Alto-Bajo real"))
display(pretty_money_table(socio_gap, money_cols=["mediana_alta", "mediana_baja", "diferencia_absoluta"]))

plot_socio = socio_pc_real[socio_pc_real["est_socio_desc"].isin(["Alto", "Bajo"])].copy()
fig, ax = plt.subplots(figsize=(9.5, 5.2))
for label, color in [("Alto", "#5B8CBA"), ("Bajo", "#C84630")]:
    g = plot_socio[plot_socio["est_socio_desc"].astype(str).eq(label)]
    ax.plot(g["anio"], g["mediana_ponderada"], marker="o", label=label, color=color)
ax.set_title("Evolución real del ingreso per cápita por estrato socioeconómico")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos de 2024, ingreso trimestral per cápita")
ax.set_xticks(YEARS)
ax.legend(frameon=False)
ax.text(0.01, -0.18, "Unidad: persona. Ponderador: factor. Variable oficial est_socio_desc.", transform=ax.transAxes, fontsize=9)
figures.append(save_doc_figure(fig, "estrato_alto_bajo_real_2018_2024.png", "Evolución estrato Alto-Bajo real"))

## CDMX y zonas metropolitanas

Se reutiliza `docs/zonas_metropolitanas_prioritarias_2020.csv`, ya validado en el notebook 09. No se reconstruye la delimitación metropolitana. CDMX se trata como entidad federativa; Valle de México como zona metropolitana.

In [ ]:
persona_geo = persona_rev5[[
    "anio", "folioviv", "foliohog", "numren", "cve_ent", "cve_mun", "entidad", "municipio",
    "region_banxico", "tam_loc_desc", "est_socio_desc", WEIGHT, PC_PERSONA, PC_PERSONA_REAL, LAB_PERSONA, LAB_PERSONA_REAL,
]].copy()
persona_geo["cve_ent"] = as_code(persona_geo["cve_ent"], 2)
persona_geo["cve_mun"] = as_code(persona_geo["cve_mun"], 3)

mapa_zm = pd.read_csv(ZM_MAP_PATH, dtype={"cve_ent": "string", "cve_mun": "string", "clave_compuesta_municipio": "string"})
mapa_zm["cve_ent"] = mapa_zm["cve_ent"].astype("string").str.zfill(2)
mapa_zm["cve_mun"] = mapa_zm["cve_mun"].astype("string").str.zfill(3)
mapa_keys = mapa_zm[["cve_ent", "cve_mun", "zona_metropolitana"]].drop_duplicates()

persona_geo = persona_geo.merge(mapa_keys, on=["cve_ent", "cve_mun"], how="left")
persona_zm = persona_geo[persona_geo["zona_metropolitana"].isin(ZONAS_PRIORITARIAS)].copy()

cdmx_persona = persona_geo[persona_geo["cve_ent"].eq("09")].copy()
cdmx_metrics = weighted_summary(cdmx_persona, "anio", PC_PERSONA_REAL)
display(Markdown("### CDMX entidad: ingreso corriente per cápita real entre personas"))
display(pretty_money_table(
    cdmx_metrics[["anio", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90", "gini_ponderado_0_100"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))

met_pc = weighted_summary(persona_zm, ["anio", "zona_metropolitana"], PC_PERSONA_REAL)
met_pc = met_pc.sort_values(["anio", "zona_metropolitana"])
display(Markdown("### Zonas metropolitanas prioritarias"))
display(pretty_money_table(
    met_pc[["anio", "zona_metropolitana", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90", "gini_ponderado_0_100"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))

met_changes = []
for zona in ZONAS_PRIORITARIAS:
    vals = met_pc[met_pc["zona_metropolitana"].eq(zona)].set_index("anio")
    if 2018 in vals.index and 2024 in vals.index:
        met_changes.append({
            "zona_metropolitana": zona,
            "mediana_2018_real": vals.loc[2018, "mediana_ponderada"],
            "mediana_2024_real": vals.loc[2024, "mediana_ponderada"],
            "variacion_real_2018_2024_pct": pct_change(vals.loc[2018, "mediana_ponderada"], vals.loc[2024, "mediana_ponderada"]),
        })
met_changes = pd.DataFrame(met_changes)
display(Markdown("### Variación real metropolitana 2018-2024"))
display(pretty_money_table(met_changes, money_cols=["mediana_2018_real", "mediana_2024_real"], pct_cols=["variacion_real_2018_2024_pct"]))

In [ ]:
persona_geo["grupo_brecha_territorial"] = pd.NA
mask_metro = persona_geo["zona_metropolitana"].isin(ZONAS_PRIORITARIAS)
persona_geo.loc[mask_metro, "grupo_brecha_territorial"] = persona_geo.loc[mask_metro, "zona_metropolitana"]
mask_small_low = (
    persona_geo["grupo_brecha_territorial"].isna()
    & persona_geo["tam_loc_desc"].astype(str).str.contains(RURAL_PATTERN, na=False)
    & persona_geo["est_socio_desc"].eq("Bajo")
)
persona_geo.loc[mask_small_low, "grupo_brecha_territorial"] = SMALL_LOW_GROUP

brecha_metro_real = weighted_summary(
    persona_geo[persona_geo["grupo_brecha_territorial"].notna()],
    ["anio", "grupo_brecha_territorial"],
    PC_PERSONA_REAL,
)
brecha_metro_real["p90_p10"] = brecha_metro_real["p90"] / brecha_metro_real["p10"]

brecha_metro_2024 = brecha_metro_real[brecha_metro_real["anio"].eq(2024)].sort_values("mediana_ponderada", ascending=False)
display(Markdown("### Brecha metropolitana real, 2024"))
display(pretty_money_table(
    brecha_metro_2024[["grupo_brecha_territorial", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90", "p90_p10", "gini_ponderado_0_100"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))

small_2024_median = float(brecha_metro_2024.loc[brecha_metro_2024["grupo_brecha_territorial"].eq(SMALL_LOW_GROUP), "mediana_ponderada"].iloc[0])
comparaciones_metro_2024 = []
for _, row in brecha_metro_2024.iterrows():
    if row["grupo_brecha_territorial"] == SMALL_LOW_GROUP:
        continue
    comparaciones_metro_2024.append({
        "comparacion": f"{row['grupo_brecha_territorial']} / grupo localidad pequeña + Bajo",
        "razon_de_medianas": row["mediana_ponderada"] / small_2024_median,
        "diferencia_absoluta_real": row["mediana_ponderada"] - small_2024_median,
    })
comparaciones_metro_2024 = pd.DataFrame(comparaciones_metro_2024)
display(Markdown("### Comparación contra localidad pequeña + Bajo, 2024"))
display(pretty_money_table(comparaciones_metro_2024, money_cols=["diferencia_absoluta_real"]))

fig, ax = plt.subplots(figsize=(9.5, 5.2))
plot_groups = brecha_metro_real[brecha_metro_real["grupo_brecha_territorial"].isin(ZONAS_PRIORITARIAS + [SMALL_LOW_GROUP])].copy()
colors = {"Valle de México": "#5B8CBA", "Guadalajara": "#76B041", "Monterrey": "#D1495B", SMALL_LOW_GROUP: "#F0A202"}
for label in ZONAS_PRIORITARIAS + [SMALL_LOW_GROUP]:
    g = plot_groups[plot_groups["grupo_brecha_territorial"].eq(label)]
    short = "Loc. pequeña + Bajo" if label == SMALL_LOW_GROUP else label
    ax.plot(g["anio"], g["mediana_ponderada"], marker="o", label=short, color=colors.get(label))
ax.set_title("Ingreso per cápita real: metrópolis y grupo localidad pequeña + Bajo")
ax.set_xlabel("Año")
ax.set_ylabel("Pesos de 2024, ingreso trimestral per cápita")
ax.set_xticks(YEARS)
ax.legend(frameon=False)
ax.text(0.01, -0.18, "Unidad: persona. Ponderador: factor. Delimitación metropolitana 2020 ya validada.", transform=ax.transAxes, fontsize=9)
figures.append(save_doc_figure(fig, "metropolis_real_2018_2024.png", "Evolución metropolitana real"))

## Ingreso laboral individual positivo

Se restringe a personas con ingreso laboral individual positivo. No se usa `factor * tot_integ`, porque la base ya está a nivel persona.

In [ ]:
labor_pos = persona_rev5[numeric(persona_rev5[LAB_PERSONA]).gt(0)].copy()
labor_total_real = weighted_summary(labor_pos, "anio", LAB_PERSONA_REAL)
labor_sex_real = weighted_summary(labor_pos, ["anio", "sexo_desc"], LAB_PERSONA_REAL)
labor_region_real = weighted_summary(labor_pos, ["anio", "region_banxico"], LAB_PERSONA_REAL)
labor_edu_real = weighted_summary(labor_pos[labor_pos["nivelaprob_desc"].notna()], ["anio", "nivelaprob_desc"], LAB_PERSONA_REAL, min_n=500)

display(Markdown("### Evolución total"))
display(pretty_money_table(
    labor_total_real[["anio", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))

display(Markdown("### Sexo: brecha descriptiva no ajustada"))
display(pretty_money_table(
    labor_sex_real[["anio", "sexo_desc", "n_muestral", "poblacion_expandida", "p25", "mediana_ponderada", "p75", "p90"]],
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral", "poblacion_expandida"],
))

sex_gap_rows = []
for year in YEARS:
    men = group_median(labor_sex_real, year, "sexo_desc", "Hombre")
    women = group_median(labor_sex_real, year, "sexo_desc", "Mujer")
    sex_gap_rows.append({
        "anio": year,
        "mediana_hombres": men,
        "mediana_mujeres": women,
        "diferencia_absoluta": men - women,
        "razon_hombres_mujeres": men / women if pd.notna(women) and women != 0 else np.nan,
    })
sex_gap = pd.DataFrame(sex_gap_rows)
display(pretty_money_table(sex_gap, money_cols=["mediana_hombres", "mediana_mujeres", "diferencia_absoluta"]))

display(Markdown("### Educación validada: mediana real de ingreso laboral positivo"))
display(pretty_money_table(
    labor_edu_real[["anio", "nivelaprob_desc", "n_muestral", "p25", "mediana_ponderada", "p75", "p90"]].sort_values(["anio", "mediana_ponderada"], ascending=[True, False]).head(60),
    money_cols=["p25", "mediana_ponderada", "p75", "p90"],
    int_cols=["n_muestral"],
))

## Resumen reproducible para documentación

Se exporta un resumen JSON en `data/interim/revision_5/` para auditar los números finales usados en la documentación. Este archivo queda ignorado por git junto con los marts intermedios.

In [ ]:
def table_records(df, decimals=4):
    out = df.copy()
    for col in out.select_dtypes(include=["float", "float64", "float32"]).columns:
        out[col] = out[col].round(decimals)
    return out.to_dict(orient="records")


summary = {
    "sources": OFFICIAL_SOURCES,
    "deflactors": table_records(deflactor_table, 8),
    "variables_reales_hogar": real_cols_hogar,
    "variables_reales_persona": real_cols_persona,
    "benchmark_validation": table_records(benchmark_validation, 6),
    "gini_invariance": table_records(gini_invariance, 8),
    "national": table_records(national, 4),
    "national_changes": table_records(national_changes, 4),
    "impact_table": table_records(impact_table, 4),
    "regional_real": table_records(regional_real, 4),
    "region_changes": table_records(region_changes, 4),
    "north_south_gap": table_records(north_south_gap, 4),
    "urban_rural_gap": table_records(urban_rural_gap, 4),
    "socio_gap": table_records(socio_gap, 4),
    "cdmx_metrics": table_records(cdmx_metrics, 4),
    "met_pc": table_records(met_pc, 4),
    "met_changes": table_records(met_changes, 4),
    "brecha_metro_2024": table_records(brecha_metro_2024, 4),
    "comparaciones_metro_2024": table_records(comparaciones_metro_2024, 4),
    "labor_total_real": table_records(labor_total_real, 4),
    "labor_sex_real": table_records(labor_sex_real, 4),
    "sex_gap": table_records(sex_gap, 4),
    "validation": table_records(validation, 4),
    "same_2024": table_records(same_2024, 12),
    "figures": [Path(f).name for f in figures],
    "rows": {"mart_persona": int(len(persona_rev5)), "mart_hogar": int(len(hogar_rev5))},
    "outputs": {
        "mart_hogar_revision_5": str(REV5 / HOGAR_FILE),
        "mart_persona_revision_5": str(REV5 / PERSONA_FILE),
        "deflactor_docs": str(DEF_DOC_PATH),
        "deflactor_revision_5": str(DEF_REV5_PATH),
    },
}

SUMMARY_PATH.write_text(json.dumps(clean_for_json(summary), ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Resumen guardado: {SUMMARY_PATH}")
print(f"Figuras guardadas: {len(figures)}")
print("Run All completado sin errores.")

## Resumen de hallazgos

- Todos los montos reales quedan expresados en pesos de 2024.
- Los nominales se conservan intactos y cada variable monetaria prioritaria tiene una contraparte `_real_2024`.
- La validación externa contra INEGI reproduce de forma cercana el ingreso corriente promedio trimestral del hogar publicado en pesos de 2024.
- La caída observada en 2020 debe leerse como observación descriptiva de una edición levantada durante el periodo de pandemia COVID-19; no se atribuye causalidad en esta libreta.
- Las comparaciones temporales de niveles, diferencias absolutas y crecimiento deben usar los montos reales; las razones dentro de un mismo año y el Gini se mantienen invariantes cuando el deflactor es común dentro del año.
- La etapa 10 deja preparado el proyecto para que los modelos futuros usen ingreso real 2024 como especificación principal y conserven ingreso nominal como sensibilidad.